In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

In [2]:
train_data = pd.read_csv('../data/model/full.csv')
train_labels = train_data['redemption_status'].values
train_data = train_data.drop(['id','redemption_status'], axis=1)
valid_data = pd.read_csv('../data/model/valid.csv')
valid_labels = valid_data['redemption_status'].values
valid_data = valid_data.drop(['id','redemption_status'], axis=1)
print(train_data.shape, valid_data.shape)

(78369, 67) (22606, 67)


In [3]:
params = {}
params['label'] = train_labels
params['categorical_feature'] = ['customer_id']
params['feature_name'] = list(train_data.columns)
train_matrix = lgb.Dataset(train_data.values, **params)
params = {}
params['label'] = valid_labels
params['categorical_feature'] = ['customer_id']
params['feature_name'] = list(valid_data.columns)
valid_matrix = lgb.Dataset(valid_data.values, **params)

In [4]:
booster = {}
booster['boosting_type'] = 'gbdt'
booster['objective'] = 'binary'
booster['learning_rate'] = 0.01
booster['num_leaves'] = 48
booster['max_depth'] = 6
booster['max_bin'] = 256
booster['subsample'] = 0.5
booster['subsample_freq'] = 1
booster['colsample_bylevel'] = 0.5
booster['colsample_bytree'] = 0.5
booster['min_split_gain'] = 0.0
booster['min_sum_hessian'] = 1
booster['nthread'] = 3
booster['verbose'] = 0
booster['metric'] = 'auc'

In [5]:
params = {}
params['params'] = booster
params['train_set'] = train_matrix
params['valid_sets'] = [train_matrix, valid_matrix]
params['num_boost_round'] = 350
params['early_stopping_rounds'] = 350
params['verbose_eval'] = 25

In [6]:
model = lgb.train(**params)

/home/ubuntu/anaconda3/lib/python3.7/site-packages/lightgbm/basic.py:1243: UserWarning: Using categorical_feature in Dataset.
  warnings.warn('Using categorical_feature in Dataset.')


Training until validation scores don't improve for 350 rounds
[25]	training's auc: 0.983852	valid_1's auc: 0.984099
[50]	training's auc: 0.988352	valid_1's auc: 0.98811
[75]	training's auc: 0.990217	valid_1's auc: 0.989985
[100]	training's auc: 0.991616	valid_1's auc: 0.991328
[125]	training's auc: 0.992723	valid_1's auc: 0.992495
[150]	training's auc: 0.993742	valid_1's auc: 0.993611
[175]	training's auc: 0.994446	valid_1's auc: 0.994356
[200]	training's auc: 0.995014	valid_1's auc: 0.99499
[225]	training's auc: 0.995522	valid_1's auc: 0.995572
[250]	training's auc: 0.995991	valid_1's auc: 0.996059
[275]	training's auc: 0.996345	valid_1's auc: 0.996411
[300]	training's auc: 0.996662	valid_1's auc: 0.996702
[325]	training's auc: 0.996949	valid_1's auc: 0.996974
[350]	training's auc: 0.997228	valid_1's auc: 0.997238
Did not meet early stopping. Best iteration is:
[350]	training's auc: 0.997228	valid_1's auc: 0.997238


In [7]:
model.save_model('../data/model/lightgbm_v2.model')

In [8]:
importance = model.feature_importance(importance_type='gain')
importance = pd.DataFrame(importance, columns=['importance'])
importance['feature'] = list(valid_data.columns)
importance['importance'] = importance['importance'] / importance['importance'].max()
importance = importance[['feature', 'importance']]
importance = importance.sort_values(by='importance', ascending=False)
importance = importance.reset_index(drop=True)

In [9]:
importance.head(10)

,feature,importance
0,customer_id,1.000000
1,cust_coup_prc,0.458902
2,sum_trx_cust_coup_price,0.376948
3,cust_cdsc_cnt,0.364048
4,cust_cdsc_sum,0.269882
5,max_trx_cust_coup_price,0.250543
6,cnt_coup_cdsc,0.248485
7,cust_coup_qty,0.233852
8,cust_cdsc,0.216784
9,sum_coup_cdsc,0.193849


In [10]:
score_data = pd.read_csv('../data/model/test.csv')
score_driver = score_data[['id']].copy()
score_data = score_data.drop(['id'], axis=1)
model = lgb.Booster(model_file='../data/model/lightgbm_v2.model')
score_driver['redemption_status'] = model.predict(score_data)
score_driver.to_csv('../data/score/score_v2.csv', index=False)

In [11]:
score_driver.shape

(50226, 2)